# ProtoFormer — Google Colab driver (PASCAL-5i)

Runtime → Change runtime type → **GPU** (T4 / L4 / A100).

Order: mount Drive → clone → install → data → symlinks → verify → train → (resume) → tensorboard → test.
Re-run cells 1, 2, 3, 5 every new session; Drive keeps data + runs.

In [ ]:
# 1. Mount Drive + GPU check
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| bf16', torch.cuda.is_bf16_supported())

In [ ]:
# 2. Clone code
%cd /content
!rm -rf protoF
!git clone https://github.com/pratamabintang/protoF.git
%cd /content/protoF
!ls data/splits/pascal/trn

In [ ]:
# 3. Install deps. Do NOT reinstall torch (breaks Colab's CUDA build).
!pip -q install gdown tensorboardX
import torch, torchvision, tensorboardX
print(torch.__version__, torchvision.__version__, tensorboardX.__version__)

## 4. Data (first session only — stored on Drive)

Downloads VOC2012, SegmentationClassAug masks, and the caffe-style `resnet50_v2.pth` init backbone.

In [ ]:
# 4. Data download to Drive (skip if already present)
!mkdir -p /content/drive/MyDrive/protoF_data
%cd /content/drive/MyDrive/protoF_data

# VOC2012 images/masks
![ -d VOC2012/JPEGImages ] || { wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar && tar -xf VOCtrainval_11-May-2012.tar && mv VOCdevkit/VOC2012 . && rm -rf VOCdevkit VOCtrainval_11-May-2012.tar; }

# SegmentationClassAug masks (HSNet file id). Fallback dropbox link in README.
![ -d VOC2012/SegmentationClassAug ] || { gdown 10zxG2VExoEZUeyQl_uXga2OWHjGeZaf2 -O aug.zip && unzip -q aug.zip -d VOC2012/ && rm aug.zip; }

# resnet50_v2.pth init backbone (PFENet Drive folder)
%cd /content/protoF && !mkdir -p initmodel
![ -f initmodel/resnet50_v2.pth ] || gdown --folder 1Hrz1wOxOZm4nIIS7UMJeL79AQrdvpj6v -O initmodel
!find initmodel -name 'resnet50_v2.pth'

In [ ]:
# 5. Per-session symlinks: data + logs on Drive
!ln -sfn /content/drive/MyDrive/protoF_data /content/data
!mkdir -p /content/drive/MyDrive/protoF_runs
%cd /content/protoF
!rm -rf logs && ln -sfn /content/drive/MyDrive/protoF_runs logs
!ls /content/data/VOC2012 | head

In [ ]:
# 6. Verify dataset completeness against repo splits
import os
root = '/content/data/VOC2012'
print('imgs', len(os.listdir(f'{root}/JPEGImages')))
print('aug masks', len(os.listdir(f'{root}/SegmentationClassAug')))
missing = []
for f in ('trn', 'val'):
    for fold in range(4):
        for line in open(f'/content/protoF/data/splits/pascal/{f}/fold{fold}.txt'):
            n = line.strip().split('__')[0]
            if not os.path.exists(f'{root}/JPEGImages/{n}.jpg'): missing.append(n + '.jpg')
            if not os.path.exists(f'{root}/SegmentationClassAug/{n}.png'): missing.append(n + '.png')
print('missing', len(set(missing)))

## 7. Train

`--amp` auto-selects dtype (T4→float16, A100/L4→bfloat16). `best_model.pt` + `last_ckpt.pt` land in `logs/<logpath>/` (on Drive).

In [ ]:
# 7. Train (fresh run)
%cd /content/protoF
!python train.py --datapath /content/data --benchmark pascal --fold 0 \
  --bsz 8 --lr 1e-3 --niter 30 --nworker 4 --nworker_val 2 --amp \
  --logpath pascal_fold0_r50_1shot

In [ ]:
# 8. Resume after a disconnect (same --logpath)
%cd /content/protoF
!python train.py --datapath /content/data --benchmark pascal --fold 0 \
  --bsz 8 --lr 1e-3 --niter 30 --nworker 4 --nworker_val 2 --amp \
  --logpath pascal_fold0_r50_1shot \
  --resume logs/pascal_fold0_r50_1shot/last_ckpt.pt

In [ ]:
# 9. Tensorboard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/protoF_runs

In [ ]:
# 10. Test
%cd /content/protoF
!python test_proto.py --datapath /content/data --benchmark pascal --fold 0 --shot 1 \
  --load logs/pascal_fold0_r50_1shot/best_model.pt \
  --bsz 1 --nworker 0 --visualize ''

## Notes

- Must run from `/content/protoF`: splits are resolved as relative `data/splits/...`.
- `initmodel/resnet50_v2.pth` must be the caffe-style deep-stem ResNet (PFENet folder), not torchvision `resnet50-19c8e357.pth`.
- `test_proto.py --visualize ''` disables writing `./vis/` dumps; drop the flag to enable.
- 5-shot test: train with `--shot 5`, then test with `--shot 5`.
- Free-tier disconnects without warning; `--resume` continues from the last epoch.